# Notebook de Demonstração - CI/CD com GitHub Actions e MLFlow
**Aula 06: Automação de Pipelines de Machine Learning**

Este notebook demonstra como o pipeline de CI/CD funciona localmente antes de ser automatizado no GitHub Actions.

## 1. Importar Bibliotecas e Configurar Ambiente

In [4]:
import warnings
import pandas as pd
import mlflow
from train import load_and_prepare_data, create_pipeline, BASELINE_PARAMS, OPTIMIZED_PARAMS
from preprocessing import CategoricalEncoder, FeatureEngineer, MissingValueImputer

warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas com sucesso!")
print(f"MLflow version: {mlflow.__version__}")

✓ Bibliotecas importadas com sucesso!
MLflow version: 3.6.0


## 2. Rodar Testes (Passo 1 do Pipeline CI/CD)

Primeiro, vamos validar que todos os componentes estão funcionando corretamente.

In [5]:
!python test_pipeline.py

test_categorical_encoder (__main__.TestPreprocessing.test_categorical_encoder)
Testa encoding categórico. ... ✓ Test CategoricalEncoder: OK
ok
test_feature_engineer (__main__.TestPreprocessing.test_feature_engineer)
Testa criação de features derivadas. ... ✓ Test FeatureEngineer: OK
ok
test_missing_value_imputer (__main__.TestPreprocessing.test_missing_value_imputer)
Testa imputação de valores faltantes. ... ✓ Test MissingValueImputer: OK
ok
test_pipeline_consistency (__main__.TestPreprocessing.test_pipeline_consistency)
Testa que o pipeline retorna colunas consistentes. ... ✓ Test Pipeline Consistency: OK
ok
test_baseline_params (__main__.TestTrainingScript.test_baseline_params)
Testa que hiperparâmetros baseline estão definidos. ... ✓ Test Baseline Params: OK
ok
test_create_pipeline (__main__.TestTrainingScript.test_create_pipeline)
Testa criação do pipeline. ... ✓ Test Create Pipeline: OK
ok
test_load_data (__main__.TestTrainingScript.test_load_data)
Testa carregamento de dados. ...

## 3. Comparar Hiperparâmetros: Baseline vs Otimizado

Vamos visualizar as diferenças entre os hiperparâmetros.

In [6]:
import pandas as pd

# Criar DataFrame comparativo
comparison = pd.DataFrame({
    'Parâmetro': list(BASELINE_PARAMS.keys()),
    'Baseline': list(BASELINE_PARAMS.values()),
    'Otimizado': [OPTIMIZED_PARAMS.get(k, 'N/A') for k in BASELINE_PARAMS.keys()]
})

print("="*70)
print("COMPARAÇÃO DE HIPERPARÂMETROS")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

COMPARAÇÃO DE HIPERPARÂMETROS
        Parâmetro Baseline  Otimizado
     n_estimators      100        124
        max_depth     None         15
min_samples_split        2         10
 min_samples_leaf        1          1
     max_features     sqrt          2
     random_state       42         42


## 4. Treinar Modelo Baseline (Passo 2 do Pipeline CI/CD)

Vamos treinar o modelo com hiperparâmetros padrão.

In [ ]:
from train import train_model

# Treinar modelo baseline
baseline_metrics = train_model(
    model_type='baseline',
    data_path='../data/heart_disease_uci.csv',
    mlflow_experiment='heart-disease-cicd-demo',
    min_accuracy=0.75
)

## 5. Treinar Modelo Otimizado (Passo 3 do Pipeline CI/CD)

Agora vamos treinar com hiperparâmetros otimizados da Aula 03.

In [ ]:
# Treinar modelo otimizado
optimized_metrics = train_model(
    model_type='optimized',
    data_path='../data/heart_disease_uci.csv',
    mlflow_experiment='heart-disease-cicd-demo',
    min_accuracy=0.75
)

## 6. Comparar Métricas (Passo 4 do Pipeline CI/CD)

Vamos comparar as métricas dos dois modelos.

In [ ]:
import pandas as pd

# Criar DataFrame comparativo
metrics_df = pd.DataFrame({
    'Métrica': list(baseline_metrics.keys()),
    'Baseline': list(baseline_metrics.values()),
    'Otimizado': list(optimized_metrics.values()),
    'Diferença': [optimized_metrics[k] - baseline_metrics[k] for k in baseline_metrics.keys()]
})

print("\n" + "="*80)
print("COMPARAÇÃO DE MÉTRICAS: BASELINE vs OTIMIZADO")
print("="*80)
print(metrics_df.to_string(index=False))
print("="*80)

# Determinar qual modelo é melhor
if optimized_metrics['test_accuracy'] > baseline_metrics['test_accuracy']:
    print("\n✓ Modelo OTIMIZADO é melhor! (Será promovido para Production)")
else:
    print("\n✗ Modelo BASELINE é melhor ou igual.")

## 7. Visualizar Resultados no MLflow UI

Execute o comando abaixo no terminal para abrir o MLflow UI e visualizar os experimentos:

```bash
mlflow ui
```

Depois acesse: http://localhost:5000

No MLflow UI você poderá:
- ✅ Comparar métricas lado a lado
- ✅ Ver hiperparâmetros de cada run
- ✅ Verificar versões registradas no Model Registry
- ✅ Checar qual modelo tem o alias "Production"

---

## 📋 Resumo do Pipeline CI/CD

Este notebook demonstrou os 4 passos principais do pipeline automatizado:

1. **Testes**: Validação de transformers e funções
2. **Treino Baseline**: Modelo com hiperparâmetros padrão
3. **Treino Otimizado**: Modelo com hiperparâmetros tunados (Aula 03)
4. **Comparação e Promoção**: Se otimizado for melhor → Production

No GitHub Actions, este processo acontece automaticamente a cada push! 🚀

---

## 🎯 Próximos Passos para Alunos

1. **Testar localmente**: Execute as células acima
2. **Modificar hiperparâmetros**: Altere valores em `train.py`
3. **Fazer commit**: `git commit -m "[train-optimized] Nova versão"`
4. **Ver pipeline em ação**: Acompanhe no GitHub Actions
5. **Comparar no MLflow**: Visualize métricas de todas as versões